In [ ]:
!pip install pdfplumber, tabula-py, pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 45.3 MB/s eta 0:00:00


In [ ]:
#sum of no of students as per approved intake = S
# no of depts = dept
#no of PCs = PC
#total Divisions = D
Titles_1 = 100  #For FE
Titles_2 = 50 * dept * 3
Volumes_2 = 10 * Titles_2 * D

#principle = A
A= 1
B = S/ (20*9)    #professors
C = (S*2) / (20*9)       #associate professors
E = (S*6)/ (20*9)        #assistant professors
Total = S / 20         #total faculty

classrooms = D * 0.5
tut = 0.25 * classrooms
labs_fe = 4      # upto intake of 600
labs = 2 * dept *3    #upto instake of 180
workshop = 1   #upto 600 intake ....... +1 for more than that

cad = 1   #upto 600 intake ....... +1 for more than that
seminar_hall = 1
library = 1
reading_hall = 1

carpet_area_classrooms_labs > 66
carpet_area_workshop > 200
carpet_area_cad > 132
carpet_area_seminar_hall > 132
carpet_area_library > 400

administrative_area > 750
amenities_area > 500


CODE TO EXTRACT DATA FROM MANDATORY DISCLOSURE PDF AND SAVE IN EXCEL

In [ ]:
import pdfplumber
import pandas as pd
from google.colab import files

# File upload functionality for Google Colab
print("Please upload your PDF file...")
uploaded = files.upload()

# Assuming the uploaded file is the first one in the dictionary
pdf_path = list(uploaded.keys())[0]
output_excel_path = "/content/Extracted_Tables_with_Titles.xlsx"

# Keywords and associated sheet titles
table_titles = {
    "Professor": "Faculty Information",
    "Classroom": "Classroom Details",
    "Laboratory": "Lab Information",
    "Course": "Courses Offered",
    "Intake": "Student Intake",
    "PCs": "PC Details",
    "titles": "Library Details",
}

# DataFrame to store extracted data
tables_with_titles = []

# Open the PDF and extract tables
with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        try:
            tables = page.extract_tables()
            for table in tables:
                # Convert table to DataFrame
                df = pd.DataFrame(table)
                # Skip tables with fewer than 3 rows or columns
                if df.shape[0] < 3 or df.shape[1] < 3:
                    continue
                # Check if the table contains any of the keywords
                for keyword, title in table_titles.items():
                    if df.astype(str).apply(lambda x: x.str.contains(keyword, case=False, na=False)).any().any():
                        # Skip large tables for specific titles
                        if title == "Courses Offered" and df.shape[0] > 20:
                            print(f"Skipping large 'Courses Offered' table on page {page_num}.")
                            continue
                        if title == "Library Details" and df.shape[0] > 20:
                            print(f"Skipping large 'Library Details' table on page {page_num}.")
                            continue
                        df["Source_Page"] = page_num  # Add source page number
                        tables_with_titles.append((df, title))
                        break
        except Exception as e:
            print(f"Error on page {page_num}: {e}")

# Save relevant tables to Excel
if tables_with_titles:
    with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
        # Use a dictionary to keep track of titles and their counts
        title_counts = {}
        for table, title in tables_with_titles:
            # Increment title count or initialize to 1
            title_counts[title] = title_counts.get(title, 0) + 1

            # Create a unique sheet name based on the title and count
            sheet_name = f"{title} ({title_counts[title]})" if title_counts[title] > 1 else title

            # Excel sheet names are max 31 chars
            table.to_excel(writer, sheet_name=sheet_name[:31], index=False, header=False)

    print(f"Extracted tables saved to: {output_excel_path}")

    # Offer file for download
    files.download(output_excel_path)
else:
    print("No relevant tables found.")


Please upload your PDF file...


Saving Mandatory Disclosure2.pdf to Mandatory Disclosure2.pdf
Skipping large 'Courses Offered' table on page 17.
Skipping large 'Courses Offered' table on page 18.
Extracted tables saved to: /content/Extracted_Tables_with_Titles.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FOR INTAKE AND DEPTS

In [32]:
# Install dependencies
!pip install pandas

# Import libraries
import pandas as pd
from google.colab import files

# Upload the file
uploaded = files.upload()

# Set the file path dynamically based on the uploaded file
file_path = list(uploaded.keys())[0]
print(f"Uploaded file: {file_path}")

# Load the Excel file and get the first sheet
excel_data = pd.ExcelFile(file_path)
first_sheet_df = pd.read_excel(file_path, sheet_name=excel_data.sheet_names[0])

# Drop rows where the 2rd column (index 2) contains 'PG'
# We use .str to handle the case where the values might be strings
filtered_df = first_sheet_df[first_sheet_df.iloc[:, 1] != 'PG']

# Get values from the 4th column of the filtered data
fourth_column_values = filtered_df.iloc[:, 3].dropna().tolist()  # Drop NaN values if any

student_intake = filtered_df.iloc[:, 3].sum() #total student intake

dept = filtered_df.shape[0]      #number of depts

Saving Extracted_Tables_with_Titles (4).xlsx to Extracted_Tables_with_Titles (4) (24).xlsx
Uploaded file: Extracted_Tables_with_Titles (4) (24).xlsx
Filtered data (after dropping rows with 'PG' in the 3rd column):
   Sr Level                                             Course  \
0   1    UG                               Computer Engineering   
1   2    UG    Electronics and\nTelecommunication\nEngineering   
2   3    UG                             Information Technology   
3   4    UG  Artificial Intelligence and Data\nScience (AI&DS)   
4   5    UG              Electronics and Computer\nEngineering   

   Number\nof Seats  
0               240  
1               240  
2               180  
3                60  
4                60  
Values in the 4th column after filtering:
[240, 240, 180, 60, 60]
Sum of values in the 4th column after filtering and converting to numeric:
780
No of depts: 5


NO OF FACULTY MEMBERS

In [ ]:
from google.colab import files
import pandas as pd

def count_professorial_ranks_in_third_column():
    # Upload file
    print("Please upload your Excel file:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    # Get the filename of the uploaded file
    filename = list(uploaded.keys())[0]

    try:
        # Load the Excel file
        excel_file = pd.ExcelFile(filename)

        # Initialize totals
        total_professors = 0
        total_associate_professors = 0
        total_assistant_professors = 0

        # Find sheets with "Faculty Information" in their title
        faculty_sheets = [sheet for sheet in excel_file.sheet_names if "Faculty Information" in sheet]

        if not faculty_sheets:
            print("No sheets found with 'Faculty Information' in the name.")
            return

        # Process each relevant sheet
        for sheet_name in faculty_sheets:
            df = pd.read_excel(filename, sheet_name=sheet_name)

            # Check if the sheet has at least 3 columns
            if df.shape[1] < 3:
                print(f"Sheet '{sheet_name}' does not have enough columns to scan the third column.")
                continue

            # Focus on the third column
            third_column = df.iloc[:, 2].astype(str).str.strip().str.lower()

            # Count exact matches
            for value in third_column:
                # Exact match checking (case insensitive)
                if value == "professor":
                    total_professors += 1
                elif value in {"associate professor", "asso.professor"}:
                    total_associate_professors += 1
                elif value in {"assistant professor", "asst professor", "asstt.professor"}:
                    total_assistant_professors += 1
                elif value == "principal":
                  print("Principal:1")

        # Display the totals
        print("\nResults:")
        print(f"Total Professors: {total_professors}")
        print(f"Total Associate Professors: {total_associate_professors}")
        print(f"Total Assistant Professors: {total_assistant_professors}")

    except Exception as e:
        print(f"An error occurred: {e}")

# Run the function
count_professorial_ranks_in_third_column()

if(total_professors >= student-intake/180):
  print("Approved no of professors")
else:
  print("The college needs to recruit more professors.College needs to recruit", (student_intake/180)-total_professors, "more professors")

if(total_associate_professors >= student_intake/90):
  print("Approved no of associate professors")
else:
  print("The college needs to recruit more associate professors.College needs to recruit", (student_intake/90)-total_associate_professors, "more associate professors")

if(total_assistant_professors >= student_intake/30):
  print("Approved no of assistant professors")
else:
  print("The college needs to recruit more assistant professors.College needs to recruit", (student_intake/30)-total_assistant_professors, "more assistant professors")

if(faculty >= student_intake/20):
  print("Approved no of faculty")
else:
  print("The college needs to recruit more faculty.College needs to recruit", (student_intake/20)-faculty, "more faculty")



Please upload your Excel file:


Saving Extracted_Tables_with_Titles (4).xlsx to Extracted_Tables_with_Titles (4) (5).xlsx
Principal:1
Principal:1
Principal:1

Results:
Total Professors: 7
Total Associate Professors: 30
Total Assistant Professors: 244


INFRASTRUCTURE

In [ ]:
from google.colab import files
import pandas as pd

def count_professorial_ranks_in_third_column():
    # Upload file
    print("Please upload your Excel file:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    # Get the filename of the uploaded file
    filename = list(uploaded.keys())[0]

    try:
        # Load the Excel file
        excel_file = pd.ExcelFile(filename)

        # Initialize totals
        total_labs = 0
        total_classrooms = 0
        total_dept_library = 0
        workshops = 0
        smart_classroom = 0

        # Find sheets with "Faculty Information" in their title
        faculty_sheets = [sheet for sheet in excel_file.sheet_names if "Classroom Details" in sheet]

        if not faculty_sheets:
            print("No sheets found with 'Classroom Details' in the name.")
            return

        # Process each relevant sheet
        for sheet_name in faculty_sheets:
            df = pd.read_excel(filename, sheet_name=sheet_name)

            # Check if the sheet has at least 3 columns
            if df.shape[1] < 3:
                print(f"Sheet '{sheet_name}' does not have enough columns to scan the third column.")
                continue

            # Focus on the third column
            third_column = df.iloc[:, 2].astype(str).str.lower()

            # Count occurrences
            lab_count = third_column.str.contains(r'\blaboratory\b', na=False).sum()
            classroom_count = third_column.str.contains(r'\bclassroom\b|\basso professor\b', na=False).sum()
            dept_library_count = third_column.str.contains(r'\bdept. library\b|\bdepartment library\b', na=False).sum()
            workshop_count = third_column.str.contains(r'\bworkshop\b', na=False).sum()
            smart_classroom_count = third_column.str.contains(r'\bsmart classroom\b', na=False).sum()

            # Update totals
            total_labs += lab_count
            total_classrooms += classroom_count
            total_dept_library += dept_library_count
            workshops += workshop_count
            smart_classroom += smart_classroom_count

        # Display the totals
        print("\nResults:")
        print(f"Total labs: {total_labs}")
        print(f"Total classrooms: {total_classrooms}")
        print(f"Total Dept Libraries: {total_dept_library}")
        print(f"Total workshops: {workshops}")
        print(f"Total smart classrooms: {smart_classroom}")

    except Exception as e:
        print(f"An error occurred: {e}")

# Run the function
count_professorial_ranks_in_third_column()

Please upload your Excel file:


Saving Extracted_Tables_with_Titles (4).xlsx to Extracted_Tables_with_Titles (4) (7).xlsx

Results:
Total labs: 45
Total classrooms: 68
Total Dept Libraries: 3
Total workshops: 2
Total smart classrooms: 10


COMBINED CODE FOR EXCEL, STUDENT INTAKE AND FACULTY INFO

In [37]:
# Install dependencies
!pip install pandas

# Import libraries
import pandas as pd
from google.colab import files

# Function to process the Excel file and extract data
def process_excel_file():
    # Upload the file
    uploaded = files.upload()

    # Check if a file is uploaded
    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    # Get the filename of the uploaded file
    file_path = list(uploaded.keys())[0]
    print(f"Uploaded file: {file_path}")

    # Load the Excel file
    excel_data = pd.ExcelFile(file_path)

    # Load the first sheet into a DataFrame
    first_sheet_df = pd.read_excel(file_path, sheet_name=excel_data.sheet_names[0])

    # Drop rows where the 2nd column (index 1) contains 'PG'
    filtered_df = first_sheet_df[first_sheet_df.iloc[:, 1] != 'PG']

    # Get values from the 4th column of the filtered data
    fourth_column_values = filtered_df.iloc[:, 3].dropna().tolist()  # Drop NaN values if any

    # Calculate total student intake (sum of the 4th column)
    student_intake = filtered_df.iloc[:, 3].sum()

    # Get the number of departments (rows in the filtered data)
    dept = filtered_df.shape[0]

    # Print extracted data
    print(f"Total Student Intake: {student_intake}")
    print(f"Number of Departments: {dept}")
    return filtered_df, student_intake, dept


# Function to process the PDF and extract tables and save them into Excel
def extract_and_save_tables(pdf_path, output_excel_path):
    tables_with_titles = []

    # Open the PDF and extract tables
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            try:
                tables = page.extract_tables()
                for table in tables:
                    # Convert table to DataFrame
                    df = pd.DataFrame(table)
                    # Skip tables with fewer than 3 rows or columns
                    if df.shape[0] < 3 or df.shape[1] < 3:
                        continue
                    # Check if the table contains any of the keywords
                    for keyword, title in table_titles.items():
                        if df.astype(str).apply(lambda x: x.str.contains(keyword, case=False, na=False)).any().any():
                            # Skip large tables for specific titles
                            if title == "Courses Offered" and df.shape[0] > 20:
                                print(f"Skipping large 'Courses Offered' table on page {page_num}.")
                                continue
                            if title == "Library Details" and df.shape[0] > 20:
                                print(f"Skipping large 'Library Details' table on page {page_num}.")
                                continue
                            df["Source_Page"] = page_num  # Add source page number
                            tables_with_titles.append((df, title))
                            break
            except Exception as e:
                print(f"Error on page {page_num}: {e}")

    # Save relevant tables to Excel
    if tables_with_titles:
        with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
            # Use a dictionary to keep track of titles and their counts
            title_counts = {}
            for table, title in tables_with_titles:
                # Increment title count or initialize to 1
                title_counts[title] = title_counts.get(title, 0) + 1

                # Create a unique sheet name based on the title and count
                sheet_name = f"{title} ({title_counts[title]})" if title_counts[title] > 1 else title

                # Excel sheet names are max 31 chars
                table.to_excel(writer, sheet_name=sheet_name[:31], index=False, header=False)

        print(f"Extracted tables saved to: {output_excel_path}")

        # Offer file for download
        files.download(output_excel_path)
    else:
        print("No relevant tables found.")


# Function to count the number of faculty members
def count_rows_in_faculty_sheets(excel_file):
    # Initialize total row count
    faculty = 0

    try:
        # Find sheets with the name "Faculty Information"
        faculty_sheets = [sheet for sheet in excel_file.sheet_names if "Faculty Information" in sheet]

        if not faculty_sheets:
            print("No sheets found with 'Faculty Information' in the name.")
            return 0

        # Iterate through the faculty information sheets
        for sheet_name in faculty_sheets:
            # Read the sheet
            df = pd.read_excel(excel_file, sheet_name=sheet_name)

            # Count non-empty rows (excluding header)
            filled_rows = df.dropna(how='all').shape[0]

            faculty += filled_rows
        return faculty

    except Exception as e:
        print(f"An error occurred: {e}")
        return 0

# Function to count professorial ranks in the third column of the Faculty Information sheets
def count_professorial_ranks_in_third_column(excel_file):
    try:
        # Initialize totals
        total_professors = 0
        total_associate_professors = 0
        total_assistant_professors = 0

        # Find sheets with "Faculty Information" in their title
        faculty_sheets = [sheet for sheet in excel_file.sheet_names if "Faculty Information" in sheet]

        if not faculty_sheets:
            print("No sheets found with 'Faculty Information' in the name.")
            return 0, 0, 0

        # Process each relevant sheet
        for sheet_name in faculty_sheets:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)

            # Check if the sheet has at least 3 columns
            if df.shape[1] < 3:
                print(f"Sheet '{sheet_name}' does not have enough columns to scan the third column.")
                continue

            # Focus on the third column
            third_column = df.iloc[:, 2].astype(str).str.strip().str.lower()

            # Count exact matches
            for value in third_column:
                # Exact match checking (case insensitive)
                if value == "professor":
                    total_professors += 1
                elif value in {"associate professor", "asso.professor"}:
                    total_associate_professors += 1
                elif value in {"assistant professor", "asst professor", "asstt.professor"}:
                    total_assistant_professors += 1
                elif value == "principal":
                    print("Principal: 1")

        # Display the totals
        print("\nResults:")
        print(f"Total Professors: {total_professors}")
        print(f"Total Associate Professors: {total_associate_professors}")
        print(f"Total Assistant Professors: {total_assistant_professors}")

        return total_professors, total_associate_professors, total_assistant_professors

    except Exception as e:
        print(f"An error occurred: {e}")
        return 0, 0, 0

# Function to check if the number of professors meets the requirements
def check_professor_requirements(total_professors, total_associate_professors, total_assistant_professors, faculty, student_intake):
    if total_professors >= student_intake / 180:
        print("Approved number of professors")
    else:
        print(f"The college needs to recruit more professors. College needs to recruit {int(student_intake / 180) - total_professors} more professors")

    if total_associate_professors >= student_intake / 90:
        print("Approved number of associate professors")
    else:
        print(f"The college needs to recruit more associate professors. College needs to recruit {int(student_intake / 90) - total_associate_professors} more associate professors")

    if total_assistant_professors >= student_intake / 30:
        print("Approved number of assistant professors")
    else:
        print(f"The college needs to recruit more assistant professors. College needs to recruit {int(student_intake / 30) - total_assistant_professors} more assistant professors")

    if faculty >= student_intake / 20:
        print("Approved number of faculty")
    else:
        print(f"The college needs to recruit more faculty. College needs to recruit {int(student_intake / 20) - faculty} more faculty")

# Main flow
def main():
    # Step 1: Upload the Excel file (this is used for both the Excel analysis and the later PDF extraction)
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    # Get the filename of the uploaded file
    excel_filename = list(uploaded.keys())[0]

    # Step 2: Process the uploaded Excel file
    filtered_df, student_intake, dept = process_excel_file()

    # Step 3: Extract and save tables from the PDF file
    # You will need to replace this with the actual PDF path for this part
    pdf_filename = "/path/to/your/pdf.pdf"
    output_excel_path = "/path/to/output/excel.xlsx"  # Specify the output Excel file path
    extract_and_save_tables(pdf_filename, output_excel_path)

    # Step 4: Load the generated Excel file
    excel_file = pd.ExcelFile(output_excel_path)

    # Step 5: Count faculty members
    faculty = count_rows_in_faculty_sheets(excel_file)

    # Step 6: Count professorial ranks
    total_professors, total_associate_professors, total_assistant_professors = count_professorial_ranks_in_third_column(excel_file)

    # Step 7: Check if the required number of professors meets the requirements
    check_professor_requirements(total_professors, total_associate_professors, total_assistant_professors, faculty, student_intake)

# Run the main flow
main()


Saving Mandatory Disclosure2.pdf to Mandatory Disclosure2 (3).pdf


KeyboardInterrupt: 

In [ ]:
!pip install farm-haystack

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.6/152.6 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.0/764.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.3/143.3 kB 10.2 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=e68ac6a7467443e4bc89c757743197781ff8e0bd6a65c363e4a2f94ceddaccc7
  Stored in directory: /root/.cache/pip/wheels/fc/ab/d4/5da2067ac95b36618c629a5f93f8094257005